# 07 — Retry Logic & Human-in-the-Loop
Explore @with_retry, retry_agent_call, and the HITL approval flow.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'

## @with_retry decorator

In [ ]:
from core.retry import with_retry
import time

attempts = []

@with_retry(max_retries=3, backoff_factor=0.01)  # fast for demo
def flaky_function(x):
    attempts.append(1)
    if len(attempts) < 3:
        raise ConnectionError(f'Attempt {len(attempts)} failed')
    return f'Success on attempt {len(attempts)}'

result = flaky_function(42)
print('Result:', result)
print('Total attempts:', len(attempts))

## retry_agent_call — returns AgentResult on failure

In [ ]:
from core.retry import retry_agent_call
from core.base_agent import AgentRequest
import core.retry as r

# Patch out sleep for speed
original_sleep = r.time.sleep
r.time.sleep = lambda _: None

call_count = [0]

def always_fails(request):
    call_count[0] += 1
    raise RuntimeError('Service unavailable')

result = retry_agent_call(always_fails, AgentRequest(query='test'), max_retries=3)
print('Success:', result.success)
print('Attempts made:', call_count[0])
print('Error message:', result.error[:50])

r.time.sleep = original_sleep  # restore

## Successful retry_agent_call

In [ ]:
from agents.information_agent import InformationAgent
from core.base_agent import AgentRequest

agent = InformationAgent()
req = AgentRequest(query='What is GRR?', data_products=['retention'])
result = retry_agent_call(agent.execute, req, max_retries=3)
print('Success:', result.success)
print('Message:', result.message)

## HITL — auto_ticket_node logic

In [ ]:
from graph.nodes import auto_ticket_node
from graph.state import initial_state
from services.jira.mock import MockJiraService
import agents.capacity_agent as cap_mod

# Inject mock Jira
original_svc = None

# Test HITL with anomalies — first pass (no approval)
state = initial_state(
    query='check retention',
    anomalies=['retention: GRR 78% below threshold 85% — risk of missing targets'],
    approved=False,
    thread_id='hitl-nb-01'
)
pending_state = auto_ticket_node(state)
print('Pending action set:', pending_state.get('pending_action') is not None)
if pending_state.get('pending_action'):
    pa = pending_state['pending_action']
    print('Action:', pa.get('action'))
    print('Count:', pa.get('count'))
    print('Anomalies:', pa.get('anomalies'))

In [ ]:
# Second pass — approved, tickets created
from services.jira.mock import MockJiraService
from unittest.mock import patch

jira_svc = MockJiraService()
state_approved = {
    **pending_state,
    'approved': True,
}
with patch('agents.capacity_agent.CapacityAgent.__init__', lambda self, **kw: setattr(self, '_svc', jira_svc) or setattr(self, '_mcp_tools', [])):
    with patch('agents.capacity_agent.get_mcp_tools', return_value=[]):
        from agents.capacity_agent import CapacityAgent
        with patch.object(CapacityAgent, '__init__', lambda self, **kw: (setattr(self, '_svc', jira_svc), setattr(self, '_mcp_tools', []))[1]):
            approved_state = auto_ticket_node(state_approved)
print('Tickets in Jira service:', len(jira_svc.tickets))
print('Auto tickets in state:', approved_state.get('auto_tickets'))
print('Pending action cleared:', approved_state.get('pending_action') is None)

## Critical HITL keywords that trigger ticket creation

In [ ]:
from graph.nodes import _HITL_KEYWORDS
print('Keywords that trigger HITL:')
for kw in _HITL_KEYWORDS:
    print(f'  "{kw}"')